In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

#### Reading File in DataFrame

In [0]:
df_matches = spark.read.format('csv')\
            .option('header', 'true')\
            .option('inferSchema', 'true')\
            .load('/Workspace/Users/asyedshabeeradhnan01@gmail.com/Data-science/raw/ipl_matches.csv')

In [0]:
df_eachball_info = spark.read.format('csv')\
               .option('header', 'true')\
               .option('inferSchema', 'true')\
               .load('/Workspace/Users/asyedshabeeradhnan01@gmail.com/Data-science/raw/ipl_ball_by_ball.csv')

#### Filtering player data for seasons >= 2020

In [0]:
df_period = df_eachball_info.withColumn('season',when(col('season').rlike(r"^\d{4}/\d{2}$"),year(col('date'))).otherwise(col('season')))\
              .withColumn('season', col('season').cast(IntegerType()))\
              .filter(col('season') >= 2020)

In [0]:
df_period.display()

#### Bowler Performance Aggregations

In [0]:
df_total_overs = df_period.groupBy('bowler').agg(
   # Total Overs and Balls Bowled
   floor(round(count('over')/6)).alias('total_overs_bowled'),
   count('over').alias('total_balls_bowled'),
   sum('total_runs').alias('runs_given'),

   # Powerplay    
   floor(round(count(when(col('is_powerplay') == 1,col('over')))/6,1)).alias('overs_bowled_in_po'),
   sum(when(col('is_powerplay') == 1,col('total_runs'))).alias('runs_in_po'),
    
   # Middle Overs 
   floor(round(count(when(col('is_middle_overs') == 1,col('over')))/6,1))\
       .alias('overs_bowled_in_mo'),
   sum(when(col('is_middle_overs') == 1,col('total_runs'))).alias('runs_in_mo'),

   # Death Overs    
   floor(round(count(when(col('is_death_overs') == 1,col('over')))/6,1))\
       .alias('overs_bowled_in_do'),
   sum(when(col('is_death_overs') == 1,col('total_runs'))).alias('runs_in_do'),

   # Wickets 
   sum(when(col('is_wicket')==1,1)).alias('wickets')
   )\
    .withColumn('economy_rate'\
            ,round(try_divide(col('runs_given'),col('total_overs_bowled'))))\
    .withColumn('bowling_avg'\
            ,round(try_divide(col('runs_given'),col('wickets'))))\
    .withColumn('strike_rate'\
            ,round(try_divide(col('total_balls_bowled'),col('wickets'))))                          

In [0]:
df_total_overs.display()

#### Wides and No-Balls Metrics

In [0]:
df_wides_noballs = df_period.groupBy('bowler').agg(
                    # Overs
                    floor(round(count('over')/6)).alias('total_overs_bowled'),
                    # Wides
                    count(when(col('wides') != 0,col('wides'))).alias('wides'),
                    sum(when(col('wides') != 0,col('total_runs'))).alias('runs_in_wides'),
                    # NoBalls
                    count(when(col('noballs') != 0,col('noballs'))).alias('noballs'),
                    sum(when(col('noballs') != 0,col('total_runs'))).alias('runs_in_noballs')
                    )

In [0]:
df_wides_noballs.display()

#### Runs Conceeded in Boundaries Metrics

In [0]:
# 1. Fours Metrics
df_no_of_4s = df_period.filter(col('total_runs').isin(4))\
                          .groupBy('bowler')\
                          .agg(
                              count(col('total_runs')).alias('no_of_4s'),
                              sum(col('total_runs')).alias('runs_conceeded_in_boundaries')
                              )

# 2. Sixes Metrics                         
df_no_of_6s = df_period.filter(col('total_runs').isin(6))\
                          .groupBy('bowler')\
                          .agg(
                              count(col('total_runs')).alias('no_of_6s'),
                              sum(col('total_runs')).alias('runs_conceeded_in_sixes')
                              )

# 3. Joins & Boundary Percentage Calculation                        
df_4s_6s_info = df_no_of_4s.join(df_no_of_6s, how='inner', on='bowler')

df_boundary_calc = df_4s_6s_info.join(df_total_overs, how='inner', on='bowler')\
                               .groupBy('bowler')\
                               .agg(
                                   round(
                                        (sum('runs_conceeded_in_boundaries')+sum('runs_conceeded_in_sixes'))/sum('runs_given')*100
                                        )
                                   .alias('total_boundary_per')
                                   )\
                               .select(
                                   'bowler',
                                   'total_boundary_per'
                                   )

df_boundary_per = df_boundary_calc.join(df_4s_6s_info, how='inner', on='bowler')\
                            .select(
                                 'bowler',
                                 'no_of_4s',
                                 'runs_conceeded_in_boundaries',
                                 'no_of_6s',
                                 'runs_conceeded_in_sixes',
                                 'total_boundary_per'
                                 )
display(df_boundary_per)

#### Dot Ball Metrics

In [0]:
df_dotball_per = df_period.groupBy('bowler').agg(
                    count(when(col('total_runs')==0,1)).alias('no_of_dotballs'),
                    count(when(((col('wides')==0) & (col('noballs')==0)),col('over')))\
                        .alias('no_of_legal_balls')
                        )\
                    .withColumn('dotball_per',
                                round(
                                    (col('no_of_dotballs')/col('no_of_legal_balls'))*100)
                                )

In [0]:
df_dotball_per.display()